In [11]:
print("Indian Patent KG-RAG")

Indian Patent KG-RAG


In [12]:
import pandas as pd

file_path = "../data/raw/2011.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [13]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 39820
Columns: 35


In [14]:
for i, column in enumerate(df.columns, 1):
    print(i, "→", column)
    

1 → Application Number
2 → Title
3 → Application Date
4 → Status
5 → Publication Number
6 → Publication Date(U/S 11A)
7 → Publication Type
8 → Application Filing Date
9 → Priority Number
10 → Priority Country
11 → Priority Date
12 → Field Of Invention
13 → Classification (IPC)
14 → Inventor Name
15 → Inventor Address
16 → Inventor Country
17 → Inventor Nationality
18 → Applicant Name
19 → Applicant Address
20 → Applicant Country
21 → Applicant Nationality
22 → Application Type
23 → E-MAIL (As Per Record)
24 → ADDITIONAL-EMAIL (As Per Record)
25 → E-MAIL (UPDATED Online)
26 → PARENT APPLICATION NUMBER
27 → PARENT APPLICATION FILING DATE
28 → REQUEST FOR EXAMINATION DATE
29 → FIRST EXAMINATION REPORT DATE
30 → Date Of Certificate Issue
31 → POST GRANT JOURNAL DATE
32 → REPLY TO FER DATE
33 → PCT INTERNATIONAL APPLICATION NUMBER
34 → PCT INTERNATIONAL FILING DATE
35 → Application Status


In [15]:
for i, column in enumerate(df.columns, 1):
    if 23 <= i <= 35:
        print(i, "→", column)

23 → E-MAIL (As Per Record)
24 → ADDITIONAL-EMAIL (As Per Record)
25 → E-MAIL (UPDATED Online)
26 → PARENT APPLICATION NUMBER
27 → PARENT APPLICATION FILING DATE
28 → REQUEST FOR EXAMINATION DATE
29 → FIRST EXAMINATION REPORT DATE
30 → Date Of Certificate Issue
31 → POST GRANT JOURNAL DATE
32 → REPLY TO FER DATE
33 → PCT INTERNATIONAL APPLICATION NUMBER
34 → PCT INTERNATIONAL FILING DATE
35 → Application Status


In [16]:
missing = df.isnull().sum()

print(missing)

Application Number                          0
Title                                       0
Application Date                            0
Status                                      0
Publication Number                          0
Publication Date(U/S 11A)                   0
Publication Type                            0
Application Filing Date                     0
Priority Number                          9993
Priority Country                         9993
Priority Date                            9994
Field Of Invention                        320
Classification (IPC)                       19
Inventor Name                             167
Inventor Address                          167
Inventor Country                          163
Inventor Nationality                      163
Applicant Name                             51
Applicant Address                          51
Applicant Country                          48
Applicant Nationality                      48
Application Type                  

In [17]:
df["Application Filing Date"] = pd.to_datetime(
    df["Application Filing Date"],
    errors="coerce"
)

print(df["Application Filing Date"].dt.year.value_counts().sort_index())

Application Filing Date
2011.0    15043
Name: count, dtype: int64


In [18]:
kg_df = df[
    [
        "Publication Number",
        "Title",
        "Inventor Name",
        "Applicant Name",
        "Classification (IPC)",
        "Status",
        "Field Of Invention"
    ]
].copy()

print("KG records:", len(kg_df))
print(kg_df.head())

KG records: 39820
  Publication Number                                              Title  \
0            26/2013                               WATER LIFTING SYSTEM   
1            37/2016  CONTENT PERSONALIZATION USING FILE REPAIR REQU...   
2            37/2016  METHOD FOR TREATING FLOOR POLISH STRIPPING WAS...   
3            02/2014  CAMERA GUARANTEEING AUTHENTICITY OF VIDEOS OR ...   
4            34/2013  A SYSTEM AND METHOD FOR COSMETIC TREATMENT AND...   

                                       Inventor Name  \
0  #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...   
1  #PERKUHN  Heiko#REMBARZ  Rene#VORWERK  Marc#WI...   
2   #KAYAMORI  Satoshi#KONDO  Tetsuro#KOMODA  Susumu   
3                                  #CHOUDURY Spandan   
4  #BARTHE, PETER, G.#SLAYTON, MICHAEL, H.#MAKIN,...   

                                      Applicant Name  \
0  #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...   
1            #TELEFONAKTIEBOLAGET LM ERICSSON (publ)   
2                         

In [19]:
# Check how inventor and applicant names are stored

print("Sample Inventor values:")
print(kg_df["Inventor Name"].head(10).to_string())

print("\n" + "="*80 + "\n")

print("Sample Applicant values:")
print(kg_df["Applicant Name"].head(10).to_string())

Sample Inventor values:
0    #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...
1    #PERKUHN  Heiko#REMBARZ  Rene#VORWERK  Marc#WI...
2     #KAYAMORI  Satoshi#KONDO  Tetsuro#KOMODA  Susumu
3                                    #CHOUDURY Spandan
4    #BARTHE, PETER, G.#SLAYTON, MICHAEL, H.#MAKIN,...
5                                    #CHOUDURY Spandan
6                                    #CHOUDURY Spandan
7                                #LEISENBERG, WOLFGANG
8                             #NAINESH SUMANBHAI DESAI
9                   #SHASHANK N VAIDYA#RAKESH R. MEHTA


Sample Applicant values:
0    #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansu...
1              #TELEFONAKTIEBOLAGET LM ERICSSON (publ)
2                                      #DIVERSEY  INC.
3                                    #CHOUDURY Spandan
4                                       #ULTHERA, INC.
5                                    #CHOUDURY Spandan
6                                    #CHOUDURY Spandan
7             

In [20]:
def split_entities(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    # Remove leading/trailing #
    value = value.strip("#")
    
    # Split multiple entities
    entities = value.split("#")
    
    # Clean whitespace
    entities = [e.strip() for e in entities if e.strip()]
    
    return entities


kg_df["Inventor_List"] = kg_df["Inventor Name"].apply(split_entities)

print(kg_df[["Publication Number", "Inventor Name", "Inventor_List"]].head(10).to_string())

  Publication Number                                                   Inventor Name                                                     Inventor_List
0            26/2013  #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai  [Koladiya Bhikhubhai Devjibhai, Vaghasiya Mansukhbhai Madhabhai]
1            37/2016    #PERKUHN  Heiko#REMBARZ  Rene#VORWERK  Marc#WILLIG  Johannes  [PERKUHN  Heiko, REMBARZ  Rene, VORWERK  Marc, WILLIG  Johannes]
2            37/2016                #KAYAMORI  Satoshi#KONDO  Tetsuro#KOMODA  Susumu               [KAYAMORI  Satoshi, KONDO  Tetsuro, KOMODA  Susumu]
3            02/2014                                               #CHOUDURY Spandan                                                [CHOUDURY Spandan]
4            34/2013    #BARTHE, PETER, G.#SLAYTON, MICHAEL, H.#MAKIN, INDER RAJ, S.   [BARTHE, PETER, G., SLAYTON, MICHAEL, H., MAKIN, INDER RAJ, S.]
5            01/2014                                               #CHOUDURY Spandan          

In [21]:
kg_df["Applicant_List"] = kg_df["Applicant Name"].apply(split_entities)

print(
    kg_df[
        ["Publication Number", "Applicant Name", "Applicant_List"]
    ].head(10).to_string()
)

  Publication Number                                                  Applicant Name                                                    Applicant_List
0            26/2013  #Koladiya Bhikhubhai Devjibhai#Vaghasiya Mansukhbhai Madhabhai  [Koladiya Bhikhubhai Devjibhai, Vaghasiya Mansukhbhai Madhabhai]
1            37/2016                         #TELEFONAKTIEBOLAGET LM ERICSSON (publ)                          [TELEFONAKTIEBOLAGET LM ERICSSON (publ)]
2            37/2016                                                 #DIVERSEY  INC.                                                  [DIVERSEY  INC.]
3            02/2014                                               #CHOUDURY Spandan                                                [CHOUDURY Spandan]
4            34/2013                                                  #ULTHERA, INC.                                                   [ULTHERA, INC.]
5            01/2014                                               #CHOUDURY Spandan          

In [22]:
import re

def split_ipc(value):
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    # Split on comma or semicolon
    ipc_codes = re.split(r"[;,]", value)
    
    # Clean whitespace
    ipc_codes = [code.strip() for code in ipc_codes if code.strip()]
    
    return ipc_codes


kg_df["IPC_List"] = kg_df["Classification (IPC)"].apply(split_ipc)

print(
    kg_df[
        ["Publication Number", "Classification (IPC)", "IPC_List"]
    ].head(10).to_string()
)

  Publication Number           Classification (IPC)                           IPC_List
0            26/2013             F04F3/00; F04F1/00               [F04F3/00, F04F1/00]
1            37/2016          H04L 12/18,G06Q 30/00           [H04L 12/18, G06Q 30/00]
2            37/2016                      C02F 1/00                        [C02F 1/00]
3            02/2014  G03B19/00,G03B43/00,G03B17/00  [G03B19/00, G03B43/00, G03B17/00]
4            34/2013                      A61B 8/00                        [A61B 8/00]
5            01/2014  H04N13/00,H04N13/04,G06T15/00  [H04N13/00, H04N13/04, G06T15/00]
6            02/2014          A61B 5/05, A61B 19/00            [A61B 5/05, A61B 19/00]
7            23/2011   F27B9/14,F27B1/00,C04B35/532   [F27B9/14, F27B1/00, C04B35/532]
8            08/2012                       A61M1/00                         [A61M1/00]
9            30/2012                      B28B19/00                        [B28B19/00]


In [23]:
import networkx as nx

G = nx.MultiDiGraph()

print("Knowledge Graph initialized!")

Knowledge Graph initialized!


In [24]:
%pip install networkx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
for _, row in kg_df.iterrows():
    publication_number = str(row["Publication Number"]).strip()

    if publication_number and publication_number != "nan":
        G.add_node(
            f"PATENT:{publication_number}",
            type="Patent",
            publication_number=publication_number,
            title=row["Title"],
            status=row["Status"],
            field=row["Field Of Invention"]
        )

print("Patent nodes created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Patent nodes created!
Total nodes: 298
Total edges: 0


In [26]:
print("Total rows:", len(kg_df))
print("Unique Publication Numbers:", kg_df["Publication Number"].nunique())
print("Duplicate Publication Numbers:", kg_df["Publication Number"].duplicated().sum())

Total rows: 39820
Unique Publication Numbers: 298
Duplicate Publication Numbers: 39522


In [27]:
# Check uniqueness of important identifier columns

id_columns = [
    "Application Number",
    "Publication Number",
    "Application Date",
    "Application Filing Date",
    "Priority Number",
    "PCT INTERNATIONAL APPLICATION NUMBER"
]

for col in id_columns:
    unique_count = kg_df[col].nunique(dropna=True)
    missing_count = kg_df[col].isna().sum()
    
    print(f"{col}")
    print(f"  Unique: {unique_count}")
    print(f"  Missing: {missing_count}")
    print(f"  Total: {len(kg_df)}")
    print("-" * 50)

KeyError: 'Application Number'

In [28]:
id_columns = [
    "Application Number",
    "Publication Number",
    "Application Date",
    "Application Filing Date",
    "Priority Number",
    "PCT INTERNATIONAL APPLICATION NUMBER"
]

for col in id_columns:
    unique_count = df[col].nunique(dropna=True)
    missing_count = df[col].isna().sum()

    print(f"{col}")
    print(f"  Unique: {unique_count}")
    print(f"  Missing: {missing_count}")
    print(f"  Total: {len(df)}")
    print("-" * 50)

Application Number
  Unique: 38649
  Missing: 0
  Total: 39820
--------------------------------------------------
Publication Number
  Unique: 298
  Missing: 0
  Total: 39820
--------------------------------------------------
Application Date
  Unique: 336
  Missing: 0
  Total: 39820
--------------------------------------------------
Application Filing Date
  Unique: 132
  Missing: 24777
  Total: 39820
--------------------------------------------------
Priority Number
  Unique: 28369
  Missing: 9993
  Total: 39820
--------------------------------------------------
PCT INTERNATIONAL APPLICATION NUMBER
  Unique: 27235
  Missing: 11491
  Total: 39820
--------------------------------------------------


In [29]:
print(
    df[
        [
            "Application Number",
            "Publication Number",
            "Title"
        ]
    ].head(20).to_string()
)

   Application Number Publication Number                                                                                                                                                                                                                            Title
0          1/MUM/2011            26/2013                                                                                                                                                                                                             WATER LIFTING SYSTEM
1        1/KOLNP/2011            37/2016                                                                                                                                                                               CONTENT PERSONALIZATION USING FILE REPAIR REQUESTS
2        2/KOLNP/2011            37/2016                                                                                                                                                     METHOD FOR TR

In [30]:
import networkx as nx

G = nx.MultiDiGraph()

print("Graph reset successfully!")

Graph reset successfully!


In [31]:
for _, row in df.iterrows():

    application_number = str(row["Application Number"]).strip()

    if application_number and application_number != "nan":

        G.add_node(
            f"PATENT:{application_number}",
            type="Patent",
            application_number=application_number,
            publication_number=str(row["Publication Number"]).strip(),
            title=row["Title"],
            status=row["Status"],
            field=row["Field Of Invention"]
        )

print("Patent nodes created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Patent nodes created!
Total nodes: 38649
Total edges: 0


In [32]:
for _, row in kg_df.iterrows():

    application_number = str(
        df.loc[row.name, "Application Number"]
    ).strip()

    if application_number == "nan":
        continue

    patent_id = f"PATENT:{application_number}"

    # Make sure the patent exists
    if patent_id not in G:
        continue

    for inventor in row["Inventor_List"]:

        inventor = " ".join(inventor.split())

        if inventor:
            inventor_id = f"INVENTOR:{inventor}"

            # Create inventor node
            G.add_node(
                inventor_id,
                type="Inventor",
                name=inventor
            )

            # Create relationship
            G.add_edge(
                patent_id,
                inventor_id,
                relationship="INVENTED_BY"
            )

print("Inventor relationships created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Inventor relationships created!
Total nodes: 124429
Total edges: 119822


In [33]:
for _, row in kg_df.iterrows():

    application_number = str(
        df.loc[row.name, "Application Number"]
    ).strip()

    if application_number == "nan":
        continue

    patent_id = f"PATENT:{application_number}"

    if patent_id not in G:
        continue

    for applicant in row["Applicant_List"]:

        applicant = " ".join(applicant.split())

        if applicant:
            applicant_id = f"APPLICANT:{applicant}"

            # Create applicant node
            G.add_node(
                applicant_id,
                type="Applicant",
                name=applicant
            )

            # Create relationship
            G.add_edge(
                patent_id,
                applicant_id,
                relationship="FILED_BY"
            )

print("Applicant relationships created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Applicant relationships created!
Total nodes: 139357
Total edges: 162983


In [34]:
for _, row in kg_df.iterrows():

    application_number = str(
        df.loc[row.name, "Application Number"]
    ).strip()

    if application_number == "nan":
        continue

    patent_id = f"PATENT:{application_number}"

    if patent_id not in G:
        continue

    for ipc in row["IPC_List"]:

        ipc = " ".join(ipc.split())

        if ipc:
            ipc_id = f"IPC:{ipc}"

            # Create IPC node
            G.add_node(
                ipc_id,
                type="IPC",
                code=ipc
            )

            # Create relationship
            G.add_edge(
                patent_id,
                ipc_id,
                relationship="CLASSIFIED_AS"
            )

print("IPC relationships created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

IPC relationships created!
Total nodes: 157794
Total edges: 209797


In [35]:
for _, row in kg_df.iterrows():

    application_number = str(
        df.loc[row.name, "Application Number"]
    ).strip()

    if application_number == "nan":
        continue

    patent_id = f"PATENT:{application_number}"

    if patent_id not in G:
        continue

    status = row["Status"]

    if pd.notna(status):
        status = " ".join(str(status).split())

        if status:
            status_id = f"STATUS:{status}"

            G.add_node(
                status_id,
                type="Status",
                name=status
            )

            G.add_edge(
                patent_id,
                status_id,
                relationship="HAS_STATUS"
            )

print("Status relationships created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Status relationships created!
Total nodes: 157795
Total edges: 249617


In [36]:
for _, row in kg_df.iterrows():

    application_number = str(
        df.loc[row.name, "Application Number"]
    ).strip()

    if application_number == "nan":
        continue

    patent_id = f"PATENT:{application_number}"

    if patent_id not in G:
        continue

    field = row["Field Of Invention"]

    if pd.notna(field):
        field = " ".join(str(field).split())

        if field:
            field_id = f"FIELD:{field}"

            G.add_node(
                field_id,
                type="Field",
                name=field
            )

            G.add_edge(
                patent_id,
                field_id,
                relationship="FIELD_OF"
            )

print("Field relationships created!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Field relationships created!
Total nodes: 157820
Total edges: 289117


In [37]:
# Count nodes by type

from collections import Counter

node_types = Counter(
    data.get("type")
    for _, data in G.nodes(data=True)
)

print("Node types:")
for node_type, count in node_types.items():
    print(f"{node_type}: {count}")

Node types:
Patent: 38649
Inventor: 85780
Applicant: 14928
IPC: 18437
Status: 1
Field: 25


In [38]:
from collections import Counter

relationship_types = Counter(
    data.get("relationship")
    for _, _, data in G.edges(data=True)
)

print("Relationship types:")
for relationship, count in relationship_types.items():
    print(f"{relationship}: {count}")

Relationship types:
INVENTED_BY: 119822
FILED_BY: 43161
CLASSIFIED_AS: 46814
HAS_STATUS: 39820
FIELD_OF: 39500


In [39]:
# Get the first Patent node

patent_nodes = [
    node for node, data in G.nodes(data=True)
    if data.get("type") == "Patent"
]

sample_patent = patent_nodes[0]

print("Sample Patent:", sample_patent)
print(G.nodes[sample_patent])

Sample Patent: PATENT:1/MUM/2011
{'type': 'Patent', 'application_number': '1/MUM/2011', 'publication_number': '26/2013', 'title': 'WATER LIFTING SYSTEM', 'status': 'Published', 'field': 'MECHANICAL ENGINEERING'}


In [40]:
# Explore everything connected to this patent

print("Patent:", sample_patent)
print()

for _, target, data in G.out_edges(sample_patent, data=True):
    target_data = G.nodes[target]

    print(
        f"{data['relationship']:20} → "
        f"{target_data.get('type'):10} → "
        f"{target_data.get('name') or target_data.get('code') or target}"
    )

Patent: PATENT:1/MUM/2011

INVENTED_BY          → Inventor   → Koladiya Bhikhubhai Devjibhai
INVENTED_BY          → Inventor   → Vaghasiya Mansukhbhai Madhabhai
FILED_BY             → Applicant  → Koladiya Bhikhubhai Devjibhai
FILED_BY             → Applicant  → Vaghasiya Mansukhbhai Madhabhai
CLASSIFIED_AS        → IPC        → F04F3/00
CLASSIFIED_AS        → IPC        → F04F1/00
HAS_STATUS           → Status     → Published
FIELD_OF             → Field      → MECHANICAL ENGINEERING


In [41]:
# Find patents associated with the same inventor

inventor_name = "Koladiya Bhikhubhai Devjibhai"
inventor_id = f"INVENTOR:{inventor_name}"

print("Inventor:", inventor_name)
print()

connected_patents = []

for patent, _, data in G.in_edges(inventor_id, data=True):

    if G.nodes[patent].get("type") == "Patent":
        connected_patents.append(patent)

print("Number of connected patents:", len(connected_patents))

print("\nConnected patents:")

for patent in connected_patents[:20]:
    patent_data = G.nodes[patent]

    print(
        patent_data["application_number"],
        "→",
        patent_data["title"]
    )

Inventor: Koladiya Bhikhubhai Devjibhai

Number of connected patents: 1

Connected patents:
1/MUM/2011 → WATER LIFTING SYSTEM


In [42]:
# Find patents sharing the same IPC classification

ipc_code = "F04F3/00"
ipc_id = f"IPC:{ipc_code}"

print("IPC:", ipc_code)
print()

connected_patents = []

for _, patent, data in G.in_edges(ipc_id, data=True):

    if G.nodes[patent].get("type") == "Patent":
        connected_patents.append(patent)

print("Number of patents with this IPC:", len(connected_patents))

print("\nRelated patents:")

for patent in connected_patents[:20]:
    patent_data = G.nodes[patent]

    print(
        patent_data["application_number"],
        "→",
        patent_data["title"]
    )

IPC: F04F3/00

Number of patents with this IPC: 0

Related patents:


In [43]:
# Find IPC nodes containing F04F

ipc_nodes = [
    (node, data)
    for node, data in G.nodes(data=True)
    if data.get("type") == "IPC" and "F04F" in str(data.get("code", ""))
]

print("Matching IPC nodes:")

for node, data in ipc_nodes[:20]:
    print(node, "→", data)

Matching IPC nodes:
IPC:F04F3/00 → {'type': 'IPC', 'code': 'F04F3/00'}
IPC:F04F1/00 → {'type': 'IPC', 'code': 'F04F1/00'}
IPC:F04F5/24 → {'type': 'IPC', 'code': 'F04F5/24'}
IPC:F04F 13/00 → {'type': 'IPC', 'code': 'F04F 13/00'}
IPC:F04F7/00 → {'type': 'IPC', 'code': 'F04F7/00'}
IPC:F04F5/10 → {'type': 'IPC', 'code': 'F04F5/10'}
IPC:F04F 5/16 → {'type': 'IPC', 'code': 'F04F 5/16'}
IPC:F04F 1/18 → {'type': 'IPC', 'code': 'F04F 1/18'}


In [44]:
ipc_id = "IPC:F04F3/00"

print("IPC node:", ipc_id)
print("Exists:", ipc_id in G)

print("\nIncoming edges:")

for source, target, data in G.in_edges(ipc_id, data=True):
    print(
        source,
        "→",
        target,
        "|",
        data
    )

IPC node: IPC:F04F3/00
Exists: True

Incoming edges:
PATENT:1/MUM/2011 → IPC:F04F3/00 | {'relationship': 'CLASSIFIED_AS'}


In [45]:
ipc_id = "IPC:F04F3/00"

connected_patents = []

for source, target, data in G.in_edges(ipc_id, data=True):

    if data.get("relationship") == "CLASSIFIED_AS":
        connected_patents.append(source)

print("IPC:", ipc_id)
print("Number of connected patents:", len(connected_patents))

print("\nConnected patents:")

for patent in connected_patents:
    patent_data = G.nodes[patent]

    print(
        patent_data["application_number"],
        "→",
        patent_data["title"]
    )

IPC: IPC:F04F3/00
Number of connected patents: 1

Connected patents:
1/MUM/2011 → WATER LIFTING SYSTEM


In [46]:
# Find patents related through shared IPC classifications

sample_patent = "PATENT:1/MUM/2011"

# Get IPCs connected to the sample patent
sample_ipcs = [
    target
    for _, target, data in G.out_edges(sample_patent, data=True)
    if data.get("relationship") == "CLASSIFIED_AS"
]

print("Sample Patent:", sample_patent)
print("IPC classifications:", sample_ipcs)

# Find patents sharing these IPCs
related_patents = {}

for ipc in sample_ipcs:

    for patent, _, data in G.in_edges(ipc, data=True):

        if (
            data.get("relationship") == "CLASSIFIED_AS"
            and patent != sample_patent
        ):
            related_patents.setdefault(patent, set()).add(ipc)

print("\nRelated patents:", len(related_patents))

for patent, shared_ipcs in list(related_patents.items())[:20]:

    patent_data = G.nodes[patent]

    print(
        patent_data["application_number"],
        "→",
        patent_data["title"],
        " | Shared IPCs:",
        ", ".join(shared_ipcs)
    )

Sample Patent: PATENT:1/MUM/2011
IPC classifications: ['IPC:F04F3/00', 'IPC:F04F1/00']

Related patents: 0


In [47]:
print("Kernel is working!")

Kernel is working!


In [48]:
print("G exists:", "G" in globals())

if "G" in globals():
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())

G exists: True
Nodes: 157820
Edges: 289117


In [49]:
import pickle
from pathlib import Path

output_path = Path("../data/processed/patent_knowledge_graph.gpickle")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "wb") as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Knowledge Graph saved successfully!")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Saved at:", output_path.resolve())

Knowledge Graph saved successfully!
Nodes: 157820
Edges: 289117
Saved at: E:\Indian-patent\indian-patent-kg-rag\data\processed\patent_knowledge_graph.gpickle
